In [15]:
#import important libraries 
import numpy as np
import pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt 
from matplotlib.colors import LinearSegmentedColormap 
from pathlib import Path  

In [16]:
#import datasets 
HERE = Path(r"C:\Users\korea\Documents\Homework\Anaconda Projects\스마트_농업_시각화\ise-yolo-kit").parent
DATA = Path(r"C:\Users\korea\Documents\Homework\Anaconda Projects\스마트_농업_시각화\ise-yolo-kit\data") 
FIG = Path(r"C:\Users\korea\Documents\Homework\Anaconda Projects\스마트_농업_시각화\ise-yolo-kit\figures")
FIG.mkdir(exist_ok=True)

In [17]:
#settings 
TARGET = {"NO3": 12.0, "K": 6.0}
ION_FOR = {"minus_N": "NO3", "minus_K": "K"}
YOLO_MIN_PLANTS = 2
ONSET_MIN_PLANTS = 2
MAX_FALSE_ALERT = 0.05
MIN_SENS = 0.80 
HORIZONS = np.arange(0, 15)
THRESH = np.round(np.arange(0.05, 0.96, 0.01), 2)

In [14]:
INK, INK2, SURF, ORANGE = "#0b0b0b", "#52514e", "#fcfcfb", "#eb6834"
plt.rcParams.update({"font.family": ["Inter", "DejaVu Sans"], "font.size": 9, "axes.spines.top": False,
    "axes.spines.right": False, "axes.edgecolor": "#c9c8c2", "figure.facecolor": SURF, "axes.facecolor": SURF,
    "axes.titlelocation": "left", "axes.titleweight": "bold", "xtick.color": INK2, "ytick.color": INK2,
    "axes.labelcolor": INK2})

In [21]:
units = pd.read_csv(DATA / "1_units_and_plants.csv")
ise = pd.read_csv(DATA / "2_ise_readings.csv", parse_dates=["timestamp"])
det = pd.read_csv(DATA / "4_yolo_detections.csv", parse_dates=["timestamp"])
ref = pd.read_csv(DATA / "5_reference_truth.csv", parse_dates=["date"])
ev = pd.read_csv(DATA / "6_events_log.csv", parse_dates=["timestamp"])
flag = set(det["synthetic"].dropna()) if "synthetic" in det.columns else set ()
NOTE = ("Semi-synthetic: timelines + ISE simulated, Yolov8 scores sampled from the real model - not results" if "semi" in flag
       else "Simulated data for layout demonstration only - not experimental results." if "yes" in flag else "")

t0 = ev.loc[ev.event == "solution_change", "timestamp"].min().normalize()
treat = units.groupby("reservoir_id").treatment.first()

In [24]:
# daily score per reservoir
ise["day"] = (ise.timestamp.dt.normalize() - t0).dt.days
ise["depl"] = (1 - ise.conc_mM / ise.ion.map(TARGET)).clip(0, 1)
ise_d = ise.groupby(["reservoir_id", "ion", "day"]).depl.mean().unstack("ion")

def ise_score(rid):
    ion = ION_FOR.get(treat[rid])
    d = ise_d.loc[rid]
    return d[ion] if ion else d.max(axis=1) # control

det["day"] = (det.timestamp.dt.normalize() - t0).dt.days
det["s"] = np.where(det["class"] != "healthy", det.confidence, 0.0)

def kth(x):
    x = np.sort(x.to_numpy())[::-1]
    return x[YOLO_MIN_PLANTS - 1] if len(x) >= YOLO_MIN_PLANTS else 0.0

yolo_d = det.groupby(["reservoir_id", "day"]).s.agg(kth)

def yolo_score(rid):
    return yolo_d.loc[rid] if rid in yolo_d.index else 0.0

def fused_score(rid):
    a, b = ise_score(rid), yolo_score(rid)
    return pd.concat([a, b], axis=1).max(axis=1) # alarm when either signal is strong

ref["day"] = (ref.date.dt.normalize() - t0).dt.days
sc = ref[ref.visible_symptom_score_0to4 >= 1].groupby(["reservoir_id", "day"]).plant_id.nunique()
onset = sc[sc >= ONSET_MIN_PLANTS].reset_index().groupby("reservoir_id").day.min()

stressed = [r for r in treat.index if treat[r] in ION_FOR and r in onset.index]
controls = [r for r in treat.index if treat[r] == "control"]
if not stressed or not controls:
    raise SystemExit("Need at least one N/K-deficient reservoir with a visible onset, and one control reservoir.")

methods = {"ISE": ise_score, "Yolov8": yolo_score, "Fused (either signal)": fused_score}
cmap = LinearSegmentedColormap.from_list("seq", ["#f3f2ee", "#a9c8f0", "#2a78d6", "#0d3a73"])
fig, axes = plt.subplots(1, 3, figsize=(13, 5.4), sharey=True)
fig.subplots_adjust(left=0.06, right=0.9, top=0.78, bottom=0.13, wspace=0.12)
fig.text(0.015, 0.975, "How far into the future can each method see?", fontsize=14, weight="bold", color=INK, va="top")
fig.text(0.015, 0.93, f"Share of N/K-deficient reservoirs already alarmed, by days before visible symptoms and alarm threshold "
                    f"(n = {len(stressed)}); greyed thresholds give > {MAX_FALSE_ALERT:.0%} false alerts on controls (n = {len(controls)})", 
         fontsize=9.5, color=INK2, va="top")
if NOTE:
    fig.text(0.015, 0.012, NOTE, fontsize=7.5, color="#8a8984", style="italic")

In [27]:
row = []
for k, (ax, (name, f)) in enumerate(zip(axes, methods.items())):
    sens = np.zeros((len(THRESH), len(HORIZONS)))
    for r in stressed: 
        s = f(r)
        for j, h in enumerate(HORIZONS): 
            upto = s[s.index <= onset[r] - h]
            best = upto.max() if len(upto) else 0.0
            sens[:, j] += best >= THRESH
    sens /= len(stressed)
    ctrl = pd.concat([f(r) for r in controls])
    fa = np.array([(ctrl >= t).mean() for t in THRESH])
    ok_t = THRESH[fa <= MAX_FALSE_ALERT]
    t_min = ok_t.min() if len(ok_t) else np.nan

In [ ]:
im = ax.pcolormesh(HORIZONS, THRESH, sens * 100, cmap=cmap, vmin=0, vmax=100, shading="nearest", rasterized=True)
if sens.max() >= 0.05:
    cs = ax.contour(HORIZONS, THRESH, sens * 100, levels=[50, 80], colors=[INK2, INK], linewidths=[0.8, 1.4])
    ax.clabel(cs, fmt="%d%%", fontsize=7.5)
if not np.isnan(t_min):
    ax.axhline(t_min, color=INK, ls="--", lw=1)
    ax.axhspan(THRESH[0] - 0.005, t_min, color=SURF, alpha=0.65, lw=0)
    usable =(sens >= MIN_SENS)